# 07 — Độ phân giải cao 384: ViT, Swin, EfficientNetV2 (MILK10k)

Giữ nguyên công thức của run tốt nhất **`vit_base__D__r5_nobce__clin+derm` (test Dice 0.479)**, chỉ tăng ảnh **224 → 384**
(ViT: mỗi ảnh 24×24 = 576 patch thay vì 14×14 = 196 → thấy rõ chi tiết nhỏ của tổn thương hơn, chậm hơn ~3 lần).

| Thành phần | Giá trị (giống run 0.479) |
|---|---|
| Fusion | MoE hướng **D** (gate theo nguồn + 3 head long-tail), **d_model 256**, metadata demo + MONET |
| Imbalance | overlay **r5_nobce**: sampler q=0.5 + squash + aug mạnh + backbone_lr_mult 1.0 + drop path 0.1 |
| Huấn luyện | lr 5e-5 (backbone = head), dropout 0.4, wd 0.05, 20 epoch, batch hiệu dụng 64 |
| Khác duy nhất | **ảnh 384**; batch 8 × accum 8 + grad checkpointing để vừa VRAM T4 |

Mặc định **2 run chạy song song, mỗi GPU một backbone** (~2–2.5 giờ, ước lượng):

| Run | Backbone |
|---|---|
| `vit_base_384__D__r5_nobce__clin+derm` | ViT-B/16 **pretrain sẵn ở 384** (`augreg_in21k_ft_in1k`), attention toàn cục |
| `swin_base_384__D__r5_nobce__clin+derm` | Swin-B **pretrain sẵn ở 384** (`window12_384.ms_in22k_ft_in1k`), attention theo cửa sổ 12×12 |

Các backbone chọn được (tất cả ảnh **384**, cùng công thức D · r5_nobce · d_model 256):

| `BACKBONES` | timm | Tham số | GFLOPs / lesion | Batch × accum | lr | ~Phút / run clin+derm (T4, ước lượng) |
|---|---|---|---|---|---|---|
| `vit_base_384` | `vit_base_patch16_384.augreg_in21k_ft_in1k` | 87M | 197 | 8 × 8 | 5e-5 | 110 |
| `swin_base_384` | `swin_base_patch4_window12_384.ms_in22k_ft_in1k` | 88M | ~190 | 8 × 8 | 5e-5 | 110 |
| `swin_large_384` | `swin_large_patch4_window12_384.ms_in22k_ft_in1k` | 197M | 416 | 4 × 16 | 5e-5 | 240 |
| `efficientnetv2_m` | `tf_efficientnetv2_m.in21k_ft_in1k` | 54M | 63 | 8 × 8 | 1e-4, drop path 0.2 | 80 |
| `efficientnetv2_l` | `tf_efficientnetv2_l.in21k_ft_in1k` | 119M | 144 | 4 × 16 | 1e-4, drop path 0.2 | 160 |
| `vit_base_224to384` | backbone run 0.479 (`augreg2`, pretrain 224), nội suy lên 384 | 87M | 197 | 8 × 8 | 5e-5 | 110 |

- Batch hiệu dụng luôn 64. Run chỉ `clin` hoặc `derm` mất khoảng một nửa thời gian.
- `swin_large_384` nặng nhất: 1 run clin+derm ~4 giờ trên T4.
- EfficientNetV2 là CNN có BatchNorm: lr cao hơn (1e-4) và drop path 0.2 như khi pretrain.
- `vit_base_224to384` tách riêng tác dụng của độ phân giải; Swin không có bản này (cửa sổ 7 của bản 224 không dùng được ở 384).

### Cách chạy trên Kaggle
1. **Settings → Accelerator: GPU T4 x2**, **Internet: On**.
2. Ô *Cấu hình* đã có sẵn link repo và link dữ liệu; chỉ cần sửa `RESTORE` khi chạy tiếp.
3. **Save Version → Save & Run All (Commit)**: notebook chạy nền tối đa 12 giờ, không cần mở trình duyệt.
   Toàn bộ `outputs/` (log, bảng, file nộp) được lưu vào output của version.
4. **Chưa chạy hết lưới** (hết 12 giờ): Add Input → output của version vừa chạy, điền vào `RESTORE`,
   rồi Save Version lần nữa. Các run đã xong được bỏ qua, run dở dang được chạy lại.

Hai GPU được dùng song song: mỗi GPU chạy một run độc lập (`scripts/run_grid.py`). Checkpoint `.pt` bị xoá
sau khi sinh file nộp để không vượt giới hạn 20GB output; `test_probs.csv` được giữ lại để ensemble.

In [ ]:
# ===================== CẤU HÌNH =====================
REPO_URL = "https://github.com/trong5nhan6/fussion.git"
BRANCH = "main"
DRIVE_URL = "https://drive.google.com/file/d/1lM1nB_4pq3Vg-GwlaTMQjtus_bpbMhGX/view?usp=drive_link"  # zip dữ liệu MILK10k
DATA_SOURCE = None    # hoặc đường dẫn Kaggle Dataset, vd "/kaggle/input/milk10k" (khi đó không tải từ Drive)

# Chạy tiếp / dùng lại kết quả: gắn output của notebook trước làm Input (Add Input -> Your Work -> Notebooks),
# rồi liệt kê thư mục outputs của nó, vd "/kaggle/input/<ten-notebook>/fussion/outputs"
RESTORE = []
TIME_BUDGET_H = 11.0

# ===================== CHỌN TỔ HỢP ĐỂ TRAIN =====================
BACKBONES = ["vit_base_384", "swin_base_384"]      # GPU rảnh lấy run kế tiếp. Chọn trong: vit_base_384 | swin_base_384
                                                   #   | swin_large_384 | efficientnetv2_m | efficientnetv2_l
                                                   #   | vit_base_224to384 (backbone run 0.479, nội suy lên 384)
ARCHS     = ["D"]                                  # A | B | C | D | E (xem notebook 05)
LOSSES    = ["r5_nobce"]                           # C, D, E: sampler_q05 | r5_nobce ; A, B: ce_sqrt_inv | sampler_q05 | r5_all
VIEWS     = ["clin+derm"]                          # clin | derm | clin+derm

In [ ]:
# ===================== THAM SỐ TRAIN =====================
# Ưu tiên cao nhất: ghi đè mọi config. None = giữ giá trị của config / lưới experiments/stage5_hires.yaml
# (lưới đã đặt sẵn: moe.d_model 256, batch 8 x accum 8, grad checkpointing).
PARAMS = {
    "train.epochs": 20,
    "train.patience": 5,
    "train.lr": None,                  # None = 5e-5 (như run 0.479); r5_nobce: backbone_lr_mult 1.0 -> backbone = head
    "train.batch_size": None,          # None = 8 (384 trên T4 16GB); hết VRAM -> 4 và accum_steps 16
    "train.accum_steps": None,         # None = 8 -> batch hiệu dụng 8 x 8 = 64 như run 0.479
    "train.folds": [0],                # [0, 1, 2, 3, 4] khi chạy đủ 5 fold
    "train.full_data": False,          # True: train trên toàn bộ dữ liệu, không val, dùng epoch cuối (đặt RUN_SUFFIX)
    "data.img_size": None,             # None = 384 | 448 (chậm hơn ~1.4 lần nữa)
    "model.dropout": None,             # None = 0.4
    "moe.d_model": None,               # None = 256 (lưới) | số khác để thử
    "seed": 42,
    "predict.tta": True,
}

# Đổi tham số so với lần chạy trước -> đặt hậu tố, vd "full". Run trùng tên đã xong sẽ bị BỎ QUA.
RUN_SUFFIX = ""

## 1. Cài đặt

In [ ]:
import os, glob
from IPython.display import Markdown, display

REPO_DIR = "/kaggle/working/" + REPO_URL.rstrip("/").split("/")[-1].removesuffix(".git")
DATA_DIR = "/tmp/MILK10k"               # ngoài /kaggle/working -> dữ liệu không bị lưu vào output notebook
os.environ["MILK10K_ROOT"] = DATA_DIR   # code tự đọc biến này thay cho data.root trong config

if os.path.isdir(REPO_DIR):
    !git -C {REPO_DIR} pull --ff-only
else:
    !git clone --depth 1 -b {BRANCH} {REPO_URL} {REPO_DIR}
%cd {REPO_DIR}
!pip install -q gdown
!git log -1 --format="Commit: %h %s"

In [ ]:
!nvidia-smi --query-gpu=index,name,memory.total --format=csv
import torch
print("torch", torch.__version__, "| CUDA:", torch.cuda.is_available(), "| số GPU:", torch.cuda.device_count())
assert torch.cuda.device_count() >= 1, "Chưa bật GPU: Settings -> Accelerator -> GPU T4 x2"

## 2. Dữ liệu

In [ ]:
src = f"--source {DATA_SOURCE}" if DATA_SOURCE else f'--drive-url "{DRIVE_URL}"'
!python scripts/setup_data.py {src} --dest {DATA_DIR}

In [ ]:
# Kiểm tra nhanh toàn bộ pipeline (không cần GPU, ~1 phút)
!python -m pytest tests -q -p no:cacheprovider

## 3. Danh sách run, cấu hình thực tế và thời gian ước tính

In [ ]:
restore_arg = ("--restore " + " ".join(RESTORE)) if RESTORE else ""
for r in RESTORE:
    print(r, "->", "OK" if os.path.isdir(r) else "KHÔNG TỒN TẠI (kiểm tra lại đường dẫn Input)")

In [ ]:
import shlex, yaml

def to_set_args(params: dict) -> str:
    # Chuyển PARAMS thành chuỗi --set key=value cho scripts (bỏ qua giá trị None)
    items = []
    for k, v in params.items():
        if v is None:
            continue
        val = yaml.safe_dump(v, default_flow_style=True).strip().removesuffix("...").strip()
        items.append(shlex.quote(f"{k}={val}"))
    return ("--set " + " ".join(items)) if items else ""

set_arg = to_set_args(PARAMS)
suffix_arg = f"--suffix {RUN_SUFFIX}" if RUN_SUFFIX else ""
print("Override:", set_arg or "(không có)")
print("Hậu tố tên run:", RUN_SUFFIX or "(không có)")

In [ ]:
# C, D, E (head long-tail): bỏ run không overlay (tag "lt") -> chỉ chạy đúng các LOSSES đã chọn
select_arg = (f"--archs {','.join(ARCHS)} --overlays {','.join(LOSSES)} "
              f"--backbones {','.join(BACKBONES)} --views {','.join(VIEWS)} --skip-lt-base")

# Ước tính phút / run clin+derm trên T4 (run 0.479 ở 224 mất ~28 phút; theo GFLOPs + grad checkpointing).
# Run 1 nhánh ảnh (clin hoặc derm) ~ một nửa. 2 GPU chạy song song.
MIN_PER_RUN = {"vit_base_384": 110, "swin_base_384": 110, "swin_large_384": 240,
               "efficientnetv2_m": 80, "efficientnetv2_l": 160, "vit_base_224to384": 110}
view_w = sum(1.0 if "+" in v else 0.5 for v in VIEWS)
n_runs = len(BACKBONES) * len(ARCHS) * len(LOSSES) * len(VIEWS)
hours = sum(MIN_PER_RUN[b] for b in BACKBONES) * len(ARCHS) * len(LOSSES) * view_w / 60 / 2
print(f"tối đa {n_runs} run, ước tính ~{hours:.1f} giờ trên 2 GPU (giới hạn mỗi phiên: {TIME_BUDGET_H} giờ)")
!python scripts/run_grid.py experiments/stage5_hires.yaml --env configs/env/kaggle_t4.yaml {select_arg} {set_arg} {suffix_arg} {restore_arg} --dry-run

## 4. Train

In [ ]:
!python scripts/run_grid.py experiments/stage5_hires.yaml --env configs/env/kaggle_t4.yaml {select_arg} {set_arg} {suffix_arg} {restore_arg} --gpus auto --est-run-h 2 --time-budget-h {TIME_BUDGET_H}

## 5. Kết quả
Run `full_data` không có Dice val nên không có trong bảng; file nộp: `outputs/<run>/submission.csv`.

In [ ]:
!python scripts/summarize.py --dirs outputs > /dev/null
display(Markdown(open("outputs/summary/summary.md", encoding="utf-8").read()))

## Nén kết quả để tải về

In [ ]:
import zipfile, datetime
from pathlib import Path

zip_path = Path("/kaggle/working") / f"07_hires_outputs_{datetime.datetime.now():%Y%m%d_%H%M}.zip"
src = Path(REPO_DIR) / "outputs"
n = 0
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
    for f in sorted(src.rglob("*")):
        if f.is_file() and f.suffix != ".pt":          # không đóng gói checkpoint
            z.write(f, Path("outputs") / f.relative_to(src))
            n += 1
print(f"Đã nén {n} file -> {zip_path} ({zip_path.stat().st_size / 2**20:.1f} MB)")
print("Tải về: tab Output của version notebook (bên phải) -> {} -> Download".format(zip_path.name))